In [ ]:
# Install all dependencies
!pip install gdown pandas scikit-learn torch tqdm --quiet

#  Download MovieLens files using gdown
import gdown

file_urls = {
    'movies':  "https://drive.google.com/uc?id=1z1UUE-TRQhaiydYF-NaoLLJO9Qb0JgXW",
    'ratings': "https://drive.google.com/uc?id=1v-DFz7NKN_nqJC_KSKILHsOmG2-dLHtv",
    'links':   "https://drive.google.com/uc?id=1nCG7xhkTcU15jKZnqA6twR4IHn_cn6qe",
    'tags':    "https://drive.google.com/uc?id=12RifUdEgvsdIKY-ofJhYHi4XdUoDYOnk",
}

for name, url in file_urls.items():
    gdown.download(url, f"{name}.csv", quiet=False)

# Load and merge data
import pandas as pd

movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
links = pd.read_csv("links.csv")
tags = pd.read_csv("tags.csv")

tags['tag'] = tags['tag'].fillna('').astype(str)
tags_agg = tags.groupby("movieId")['tag'].apply(lambda x: '|'.join(set(x))).reset_index()

movie_features = movies.merge(links, on='movieId', how='left').merge(tags_agg, on='movieId', how='left')
data = ratings.merge(movie_features, on='movieId', how='left')

# Encode features
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import numpy as np

user_enc = LabelEncoder()
item_enc = LabelEncoder()
data['user_enc'] = user_enc.fit_transform(data['userId'])
data['item_enc'] = item_enc.fit_transform(data['movieId'])

data['genres'] = data['genres'].fillna('').apply(lambda x: x.split('|'))
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(data['genres'])

X = {
    'user': data['user_enc'].values,
    'item': data['item_enc'].values,
    'genre': genre_encoded
}
y = data['rating'].values.astype(np.float32)

# Train/val split using index slicing
from sklearn.model_selection import train_test_split

n_samples = len(y)
train_idx, val_idx = train_test_split(np.arange(n_samples), test_size=0.1, random_state=42)

X_train = {
    'user': X['user'][train_idx],
    'item': X['item'][train_idx],
    'genre': X['genre'][train_idx]
}
X_val = {
    'user': X['user'][val_idx],
    'item': X['item'][val_idx],
    'genre': X['genre'][val_idx]
}
y_train = y[train_idx]
y_val = y[val_idx]

# PyTorch Dataset and Model
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

class MovieLensDataset(Dataset):
    def __init__(self, X, y):
        self.user = torch.tensor(X['user'], dtype=torch.long)
        self.item = torch.tensor(X['item'], dtype=torch.long)
        self.genre = torch.tensor(X['genre'], dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.user)

    def __getitem__(self, idx):
        return self.user[idx], self.item[idx], self.genre[idx], self.y[idx]

class RecommenderNet(nn.Module):
    def __init__(self, n_users, n_items, genre_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, 64)
        self.item_emb = nn.Embedding(n_items, 64)
        self.fc1 = nn.Linear(64 + 64 + genre_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.output = nn.Linear(64, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, user, item, genre):
        u = self.user_emb(user)
        i = self.item_emb(item)
        x = torch.cat([u, i, genre], dim=1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        return self.output(x).squeeze()


train_dataset = MovieLensDataset(X_train, y_train)
val_dataset = MovieLensDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RecommenderNet(len(user_enc.classes_), len(item_enc.classes_), genre_encoded.shape[1]).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop with tqdm
for epoch in range(5):
    model.train()
    total_loss = 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} Training")

    for user, item, genre, target in train_bar:
        user, item, genre, target = user.to(device), item.to(device), genre.to(device), target.to(device)
        optimizer.zero_grad()
        preds = model(user, item, genre)
        loss = criterion(preds, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        train_bar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} - Avg Train Loss: {total_loss / len(train_loader):.4f}")

# Evaluation with tqdm
model.eval()
with torch.no_grad():
    val_loss = 0
    val_bar = tqdm(val_loader, desc="Validation")
    for user, item, genre, target in val_bar:
        user, item, genre, target = user.to(device), item.to(device), genre.to(device), target.to(device)
        preds = model(user, item, genre)
        loss = criterion(preds, target)
        val_loss += loss.item()
        val_bar.set_postfix(loss=loss.item())

    print(f"Validation MSE: {val_loss / len(val_loader):.4f}")
